# Projeto integrado. Gestão da base de conhecimento do atendimento da Quantum Commerce

Knowledge Management and Prompt Engineering. MBA AI Engineering & Multi-Agents.

**Integrantes do grupo.** [nomes completos, um por linha]

Este notebook é o esqueleto do trabalho, com as seções na ordem do enunciado. As células marcadas com `SUA VEZ` são as células do grupo. As células de análise, em markdown, vêm depois de cada medição e recebem a leitura dos números e a decisão que eles motivaram. O notebook precisa rodar do início ao fim, sem célula que dependa de ação manual além da entrada da chave pelo `getpass`.

prof. Thiago Nascimento Nogueira
e-mail: profthiago.nogueira@fiap.com.br
linkedIn: https://www.linkedin.com/in/thnogueira/

## 1. Configuração e carga dos dados

In [ ]:
# Instalação. Acrescente o que o grupo usar.
!pip install -q sentence-transformers rank_bm25 anthropic ragas

In [ ]:
from getpass import getpass
import os
os.environ["ANTHROPIC_API_KEY"] = getpass("Chave de API: ")

In [ ]:
# Os dados, carregados pelos links do enunciado.
import urllib.request, zipfile, io, json, csv

BASE = "https://raw.githubusercontent.com/thiagonogueira/datasets/main/knowledge-management/"

with urllib.request.urlopen(BASE + "documentos.zip") as r:
    zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
manifest = json.load(open("documentos/manifest.json", encoding="utf-8"))
DOCS = {k: open(f"documentos/{k}.md", encoding="utf-8").read() for k in manifest}

def baixa_csv(nome):
    with urllib.request.urlopen(BASE + nome) as r:
        return list(csv.DictReader(io.StringIO(r.read().decode("utf-8")), delimiter=";"))

chamados = baixa_csv("chamados-clientes.csv")
perguntas_atendimento = baixa_csv("perguntas-atendimento.csv")

anotados = [ch for ch in chamados if ch["trecho_referencia"]]
sem_anotacao = [ch for ch in chamados if not ch["trecho_referencia"]]
print(len(DOCS), "documentos.", len(chamados), "chamados,", len(anotados), "anotados e", len(sem_anotacao), "sem anotação.", len(perguntas_atendimento), "perguntas do atendimento.")

In [ ]:
# Um documento e um chamado, como exemplo.
k = "pol-dev-03"
print(manifest[k])
print(DOCS[k][:600])
print()
ch = anotados[0]
print(ch["id_reclamacao"], ch["classe"], ch["data_abertura"])
print(ch["texto_reclamacao"])
print("referência.", ch["documento_referencia"])

## 2. Linha de base

O assistente como a equipe anterior o construiu, um RAG naive. Chunk de tamanho fixo sobre todos os documentos, recuperação por similaridade sem filtro, geração com os trechos no prompt. O prompt traz o papel do assistente, o escopo e a instrução de responder só com o que está nos trechos.

As duas medidas do enunciado. Recuperação, a fração dos chamados anotados em que o trecho de referência aparece no que foi recuperado. Resposta, a fração dos chamados em que a resposta informa o prazo ou o valor do trecho de referência sem afirmar regra ausente dos trechos recebidos, e a fração dos chamados sem anotação em que o assistente declara que não tem a informação. O critério de julgamento das respostas é do grupo e fica registrado aqui.

As colunas `documento_referencia` e `trecho_referencia` entram só nas células de medição.

In [ ]:
# ========================== SUA VEZ ==========================
# Indexação da linha de base. Chunk de tamanho fixo sobre todos os documentos, embedding e índice.

In [ ]:
# ========================== SUA VEZ ==========================
# Recuperação e geração. Defina recupera(chamado) devolvendo a lista de trechos, e responde(chamado, trechos) devolvendo o texto da resposta.

In [ ]:
# Medida 1, recuperação. Esta célula é a mesma para todas as configurações.
# Acerto quando todos os trechos de referência do chamado aparecem no texto recuperado.
acertos = 0
erradas = []
for ch in anotados:
    texto = "\n".join(recupera(ch)).lower()
    partes = ch["trecho_referencia"].split(" [...] ")
    hit = all(p.lower() in texto for p in partes)
    acertos += hit
    if not hit:
        erradas.append(ch["id_reclamacao"])
print(f"recuperação. {acertos} de {len(anotados)}")
print("erradas.", erradas)

In [ ]:
# ========================== SUA VEZ ==========================
# Medida 2, resposta. Gere a resposta de cada chamado e julgue pelo critério do grupo. Registre a fração com prazo ou valor certo sem regra inventada, e a fração de abstenção nos chamados sem anotação.

**Análise da linha de base.** [Leitura das duas medidas. Que chamados erram e o que têm em comum. A decisão, o que o grupo vai mudar primeiro e por quê.]

## 3. Configurações testadas

Uma seção por configuração. Cada configuração muda uma coisa em relação à anterior. As duas células de medição são as mesmas da seção 2. Repita o bloco (mudança, medida 1, medida 2, análise) para cada configuração.

### Configuração 1. [nome da mudança]

In [ ]:
# ========================== SUA VEZ ==========================
# O que muda em relação à configuração anterior, e só isso.

In [ ]:
# Medida 1, recuperação, a mesma da seção 2.
acertos = 0
erradas = []
for ch in anotados:
    texto = "\n".join(recupera(ch)).lower()
    partes = ch["trecho_referencia"].split(" [...] ")
    hit = all(p.lower() in texto for p in partes)
    acertos += hit
    if not hit:
        erradas.append(ch["id_reclamacao"])
print(f"configuração 1, recuperação. {acertos} de {len(anotados)}")
print("erradas.", erradas)

In [ ]:
# ========================== SUA VEZ ==========================
# Medida 2, resposta, pelo mesmo critério da seção 2.

**Análise da configuração 1.** [As duas medidas contra a configuração anterior. Que chamados ganhou e que chamados perdeu. A decisão.]

### Configuração 2. [nome da mudança]

In [ ]:
# ========================== SUA VEZ ==========================

In [ ]:
# Medida 1, recuperação, a mesma da seção 2.

In [ ]:
# ========================== SUA VEZ ==========================
# Medida 2, resposta.

**Análise da configuração 2.** [...]

## 4. Comparação pelo RAGAS

Linha de base contra a configuração final, com o mesmo modelo e temperatura zero, sobre uma amostra dos chamados anotados.

In [ ]:
# ========================== SUA VEZ ==========================

**Análise do RAGAS.** [O que cada métrica diz sobre as duas configurações, e se a leitura concorda com as duas medidas do grupo.]

## 5. Conclusão

1. Qual configuração foi escolhida, e por quê, pelos números do próprio trabalho.
2. Onde o assistente ainda erra, e em que etapa (recuperação ou geração).
3. O que a equipe faria em seguida.

## 6. Extra. Perguntas da equipe de atendimento

As 14 perguntas de `perguntas-atendimento.csv`. Primeiro, o que a melhor configuração responde. Depois, a explicação da causa, a solução construída e a comparação.

In [ ]:
# ========================== SUA VEZ ==========================
# Respostas da melhor configuração às 14 perguntas.

**Por que as respostas saem incompletas ou erradas.** [Com base no que foi recuperado para cada pergunta.]

In [ ]:
# ========================== SUA VEZ ==========================
# A solução do grupo, e as respostas dela às 14 perguntas.

**Comparação.** [Quantas cada forma acertou, com a justificativa de cada acerto.]